# Soft / hard-sphere dilute Bose gas — results analysis

Analysis of an **SS10** equation-of-state sweep (penetrable sphere, $R=10$, scattering length $a=1$,
$N$ bosons, 3D PBC) produced by the `soft_sphere_gas` sweep machinery and stored in
`<root>/soft-hard-bosons/outputs/`.

All numbers are in **paper units** ($\hbar^2/2m=1$, $a=1$, energies in $\hbar^2/2ma^2$), so they drop
straight onto Mazzanti–Polls–Fabrocini (arXiv:cond-mat/0305502). The control parameter is the gas
parameter $x=\rho a^3$.

This notebook only **reads and analyses** an existing DB — it does not train. It reuses the shipped
machinery: `db`, `sweep.load_curve`, `analysis.plot_curve`, `dilute_gas` (Lee-Yang / $4\pi x$ / Eq.31),
and the per-run `history.csv` / `verdict.json` artifacts.

**Corridor that must hold for a converged variational run:** $4\pi x \le E/N \le E_{31}$, with
$E/N \to$ Lee-Yang as $x\to0$.


## 0. Paths & imports

The notebook lives in `<root>/qvarnet/soft_sphere_gas/` and the results live in
`<root>/soft-hard-bosons/outputs/`. Both are derived from the notebook location — edit `ROOT`
below if your layout differs.

In [1]:
from pathlib import Path
import sys, json, math, sqlite3

# --- configurable paths (defaults derived from the notebook's own location) ---
SOFT_DIR = Path.cwd()                       # .../qvarnet/soft_sphere_gas  (run the nb from here)
ROOT     = SOFT_DIR.parent.parent           # .../<root>
DATA_DIR = ROOT / 'soft-hard-bosons' / 'outputs'
DB_PATH  = DATA_DIR / 'soft_sphere.db'
RUNS_DIR = DATA_DIR / 'runs'

# make the bare-module machinery importable (db, sweep, dilute_gas, analysis, point, artifacts)
if str(SOFT_DIR) not in sys.path:
    sys.path.insert(0, str(SOFT_DIR))

assert SOFT_DIR.name == 'soft_sphere_gas', f'run this notebook from soft_sphere_gas/, not {SOFT_DIR}'
assert DB_PATH.exists(), f'DB not found: {DB_PATH}  (edit ROOT/DATA_DIR above)'
print('soft_sphere_gas :', SOFT_DIR)
print('reading results :', DATA_DIR)
print('db              :', DB_PATH)

soft_sphere_gas : /home/pfargas/Desktop/PhD/qvarnet/soft_sphere_gas
reading results : /home/pfargas/Desktop/PhD/soft-hard-bosons/outputs
db              : /home/pfargas/Desktop/PhD/soft-hard-bosons/outputs/soft_sphere.db


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import db
import sweep
import dilute_gas
import analysis
from point import Potential
import artifacts

# benchmark shorthands (paper units, a=1)
def lower_bound(x):  return 4.0 * math.pi * x                       # Lieb-Yngvason, rigorous
lee_yang = dilute_gas.lee_yang_energy_per_particle                 # Eq.1, x->0 universal limit
eq31     = dilute_gas.first_order_energy_upper_bound               # Eq.31, variational UB

## 1. Load the sweep DB

Every completed run becomes one row. We pull the scalars and unpack the convergence `verdict`
(stationarity, Geweke $z$, split-$\hat R$, epochs) into a tidy frame.

In [3]:
conn = db.connect(str(DB_PATH))
print('queue status:', db.status_counts(conn))

rows = conn.execute("SELECT * FROM runs WHERE status='done' ORDER BY x, N, seed").fetchall()
print(f'{len(rows)} done runs')

def _vget(vj, key, default=None):
    try: return json.loads(vj).get(key, default)
    except (TypeError, ValueError): return default

recs = []
for r in rows:
    recs.append(dict(
        x=r['x'], N=r['N'], seed=r['seed'], L=r['L'],
        e_per_n=r['e_per_n'], err=r['err_per_n'], sigma=r['sigma_e_per_n'],
        acceptance=r['acceptance'], passed=bool(r['passed']),
        upper_bound_db=r['upper_bound'],
        stationary=_vget(r['verdict_json'], 'stationary'),
        geweke_z=_vget(r['verdict_json'], 'geweke_z'),
        split_rhat=_vget(r['verdict_json'], 'split_rhat'),
        epochs=_vget(r['verdict_json'], 'epochs_ran'),
        run_dir=r['run_dir'],
    ))
df = pd.DataFrame(recs).sort_values('x').reset_index(drop=True)
df

queue status: {'done': 7}
7 done runs


,x,N,seed,L,e_per_n,err,sigma,acceptance,passed,upper_bound_db,stationary,geweke_z,split_rhat,epochs,run_dir
0,0.000010,64,0,185.663553,0.000140,0.000004,0.000122,0.875736,False,0.000143,True,0.816082,0.999986,2000,runs/SS10_x1.000e-05_N64_s0_87df2862
1,0.000030,64,0,128.076078,0.000425,0.000007,0.000211,0.808862,False,0.000435,True,1.419753,1.000000,2000,runs/SS10_x3.046e-05_N64_s0_87df2862
2,0.000093,64,0,88.350576,0.001288,0.000011,0.000363,0.717356,False,0.001325,False,2.938950,0.999964,2000,runs/SS10_x9.280e-05_N64_s0_87df2862
3,0.000283,64,0,60.946777,0.003909,0.000019,0.000612,0.645315,False,0.004036,False,4.566208,0.999974,2000,runs/SS10_x2.827e-04_N64_s0_87df2862
4,0.000861,64,0,42.042846,0.011890,0.000030,0.000970,0.565101,False,0.012295,True,1.644565,0.999961,2000,runs/SS10_x8.612e-04_N64_s0_87df2862
5,0.002623,64,0,29.002369,0.036333,0.000042,0.001349,0.536278,False,0.037455,True,0.003946,1.000059,2000,runs/SS10_x2.623e-03_N64_s0_87df2862
6,0.007992,64,0,20.006671,0.111626,0.000048,0.001523,0.607424,False,0.114101,True,1.463889,1.000013,2000,runs/SS10_x7.992e-03_N64_s0_87df2862


## 2. Physics corridor check

Same test as `check_db.py`, recomputing the three benchmarks independently from `dilute_gas`:
a run is inside the corridor when $4\pi x - \mathrm{err} \le E/N \le E_{31} + \mathrm{err}$.
`E/4pix` is the ratio plotted in the paper's Fig.1 (the rigorous lower bound is the line at 1).

In [4]:
R = 10.0
V0 = Potential.from_R(R).V0_paper           # V0 fixed by a=1 (Eq.10)

chk = df.copy()
chk['4pix']    = lower_bound(chk['x'])
chk['LeeYang'] = chk['x'].map(lee_yang)
chk['Eq31']    = chk['x'].map(lambda x: eq31(x, V0, R))
chk['E/4pix']  = chk['e_per_n'] / chk['4pix']
chk['E/LY']    = chk['e_per_n'] / chk['LeeYang']
chk['below_lb'] = chk['e_per_n'] < chk['4pix'] - chk['err']
chk['above_ub'] = chk['e_per_n'] > chk['Eq31'] + chk['err']
chk['corridor_ok'] = ~(chk['below_lb'] | chk['above_ub'])

cols = ['x','N','seed','e_per_n','err','4pix','LeeYang','Eq31','E/4pix','E/LY',
        'corridor_ok','stationary','geweke_z','split_rhat','epochs','passed']
n_viol = int((~chk['corridor_ok']).sum())
print(f'corridor violations: {n_viol}/{len(chk)}')
print(f'non-stationary runs: {int((chk["stationary"]==False).sum())}/{len(chk)} '
      f'(verdict.passed: {int(chk["passed"].sum())}/{len(chk)})')
chk[cols].style.format({'e_per_n':'{:.4e}','err':'{:.2e}','4pix':'{:.4e}',
    'LeeYang':'{:.4e}','Eq31':'{:.4e}','E/4pix':'{:.3f}','E/LY':'{:.3f}',
    'geweke_z':'{:.2f}','split_rhat':'{:.4f}'})

corridor violations: 0/7
non-stationary runs: 2/7 (verdict.passed: 0/7)


,x,N,seed,e_per_n,err,4pix,LeeYang,Eq31,E/4pix,E/LY,corridor_ok,stationary,geweke_z,split_rhat,epochs,passed
0,0.000010,64,0,1.4004e-04,3.80e-06,1.2566e-04,1.2758e-04,1.4277e-04,1.114,1.098,True,True,0.82,1.0000,2000,False
1,0.000030,64,0,4.2456e-04,6.60e-06,3.8281e-04,3.9298e-04,4.3492e-04,1.109,1.080,True,True,1.42,1.0000,2000,False
2,0.000093,64,0,1.2878e-03,1.13e-05,1.1662e-03,1.2203e-03,1.3249e-03,1.104,1.055,True,False,2.94,1.0000,2000,False
3,0.000283,64,0,3.9091e-03,1.91e-05,3.5525e-03,3.8401e-03,4.0361e-03,1.100,1.018,True,False,4.57,1.0000,2000,False
4,0.000861,64,0,1.1890e-02,3.03e-05,1.0822e-02,1.2351e-02,1.2295e-02,1.099,0.963,True,True,1.64,1.0000,2000,False
5,0.002623,64,0,3.6333e-02,4.22e-05,3.2968e-02,4.1097e-02,3.7455e-02,1.102,0.884,True,True,0.00,1.0001,2000,False
6,0.007992,64,0,1.1163e-01,4.76e-05,1.0043e-01,1.4366e-01,1.1410e-01,1.111,0.777,True,True,1.46,1.0000,2000,False


## 3. Equation of state — $E/N$ vs $x$ with benchmarks

`sweep.load_curve` aggregates seeds per $x$. **Note:** every run here has `passed=False`
(the early-stop target was 0, never satisfied — not a physics failure), so we pass
`require_passed=False` to include them. We mark non-stationary points so they aren't read as
physics. Left: paper's Fig.1 axis $E/N \div 4\pi x$; right: absolute $E/N$ (log-log).

In [5]:
curve = sweep.load_curve(conn, 'SS10', int(df['N'].iloc[0]), require_passed=False)
print('curve x-points:', len(curve['x']), '  seeds/x:', curve['n_per_x'])
pot = Potential.from_R(R)

fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4.4))
analysis.plot_curve(curve, pot, scaled=True,  ax=axL)
analysis.plot_curve(curve, pot, scaled=False, ax=axR)
axR.set_yscale('log')

# overlay the non-stationary points (open red circles) as a 'do not trust yet' flag
bad = chk[chk['stationary'] == False]
if len(bad):
    axL.scatter(bad['x'], bad['e_per_n']/bad['4pix'], s=120, facecolors='none',
                edgecolors='red', linewidths=1.6, zorder=5, label='non-stationary')
    axR.scatter(bad['x'], bad['e_per_n'], s=120, facecolors='none',
                edgecolors='red', linewidths=1.6, zorder=5)
    axL.legend(fontsize=8)
fig.suptitle('SS10 dilute Bose gas: VMC equation of state vs analytic benchmarks')
fig.tight_layout()
fig.savefig(DATA_DIR / 'ss10_equation_of_state.png', dpi=140)
print('saved', DATA_DIR / 'ss10_equation_of_state.png')

curve x-points: 7   seeds/x: [1, 1, 1, 1, 1, 1, 1]
saved /home/pfargas/Desktop/PhD/soft-hard-bosons/outputs/ss10_equation_of_state.png


## 4. Beyond-mean-field: is the LHY $\sqrt{x}$ correction resolved?

Mean field gives $E/N = 4\pi x\,(a_{\rm eff}/a)$, so the ratio $E/N\div4\pi x$ **is** $a_{\rm eff}/a$
if it is flat in $x$. Lee-Yang instead predicts it *rising* as $1+\tfrac{128}{15}\sqrt{x/\pi}$.
Plotting the ratio vs $\sqrt{x}$ separates the two: a horizontal band = mean-field with a renormalised
coupling; an upward slope = genuine LHY. (A flat ratio is the expected signature of a DeepSet ansatz
with **no pairwise Jastrow / correlation hole**.)

In [6]:
good = chk[chk['stationary'] != False]      # use only stationary points for the fit
ratio = good['E/4pix'].to_numpy()
w = 1.0 / (good['err'].to_numpy() / good['4pix'].to_numpy())**2
a_eff = float(np.sum(w*ratio)/np.sum(w))
a_eff_err = float(np.sqrt(1.0/np.sum(w)))
print(f'weighted-mean ratio  E/N / 4pix = a_eff/a = {a_eff:.4f} +/- {a_eff_err:.4f}'
      f'   (over {len(good)} stationary points)')

fig, ax = plt.subplots(figsize=(6.5, 4.2))
sx = np.sqrt(chk['x'].to_numpy())
ax.errorbar(sx, chk['E/4pix'], yerr=chk['err']/chk['4pix'], fmt='o', color='tab:blue',
            capsize=3, label='VMC  E/N / 4pix')
if len(bad):
    ax.scatter(np.sqrt(bad['x']), bad['e_per_n']/bad['4pix'], s=120, facecolors='none',
               edgecolors='red', linewidths=1.6, label='non-stationary')
xx = np.linspace(chk['x'].min(), chk['x'].max(), 200)
ax.plot(np.sqrt(xx), 1 + (128/15)*np.sqrt(xx/math.pi), 'k--', lw=1, label='Lee-Yang ratio')
ax.axhline(1.0, color='0.6', ls=':', lw=1, label='mean field (=1)')
ax.axhline(a_eff, color='tab:green', lw=1, label=f'fit a_eff/a = {a_eff:.3f}')
ax.set_xlabel(r'$\sqrt{x}$'); ax.set_ylabel(r'$E/N \,/\, 4\pi x$')
ax.set_title('Beyond-mean-field test: flat band (a_eff) vs LHY slope')
ax.legend(fontsize=8); fig.tight_layout()
fig.savefig(DATA_DIR / 'ss10_lhy_ratio.png', dpi=140); print('saved ss10_lhy_ratio.png')

weighted-mean ratio  E/N / 4pix = a_eff/a = 1.1101 +/- 0.0004   (over 5 stationary points)
saved ss10_lhy_ratio.png


## 5. Per-run convergence (history.csv)

$E/N$ vs epoch for each run, straight from the artifacts. Non-stationary runs (red) are still
drifting at the last epoch — those points need more epochs (or `sampler='pt'`) before they enter
any physics figure.

In [7]:
fig, ax = plt.subplots(figsize=(8, 4.6))
for _, row in chk.iterrows():
    rd = RUNS_DIR / Path(row['run_dir']).name
    hist = rd / 'history.csv'
    if not hist.exists():
        print('missing', hist); continue
    h = pd.read_csv(hist)
    nonstat = (row['stationary'] == False)
    ax.plot(h['epoch'], h['e_per_n_paper'], lw=1.2,
            color=('red' if nonstat else None),
            label=f"x={row['x']:.2e}" + ('  (non-stat)' if nonstat else ''))
ax.set_yscale('log')
ax.set_xlabel('epoch'); ax.set_ylabel('E/N  [paper units]')
ax.set_title('Training convergence per x (red = non-stationary)')
ax.legend(fontsize=7, ncol=2); fig.tight_layout()
fig.savefig(DATA_DIR / 'ss10_convergence.png', dpi=140); print('saved ss10_convergence.png')

saved ss10_convergence.png


## 6. Findings & next steps

*(auto-summary from the cells above; edit after inspecting the figures)*

In [8]:
n_done = len(chk); n_stat = int((chk['stationary'] != False).sum())
print('SS10 sweep summary')
print('-'*60)
print(f'points (done)        : {n_done}   x in [{chk["x"].min():.2e}, {chk["x"].max():.2e}]')
print(f'inside corridor      : {int(chk["corridor_ok"].sum())}/{n_done}')
print(f'stationary           : {n_stat}/{n_done}')
print(f'a_eff/a (flat ratio)  : {a_eff:.4f} +/- {a_eff_err:.4f}')
print()
print('non-stationary points to re-run (more epochs / sampler=pt):')
for _, b in chk[chk['stationary']==False].iterrows():
    print(f"   x={b['x']:.3e}  geweke_z={b['geweke_z']:.2f}  epochs={b['epochs']}")
print()
print('Interpretation:')
print(' - E/N stays inside [4pix, Eq31] -> variationally consistent.')
print(' - E/N / 4pix ~ const -> mean-field regime with renormalised coupling a_eff;')
print('   the LHY sqrt(x) correction is NOT resolved (expected: DeepSet has no pairwise Jastrow).')
print(' - To probe beyond-mean-field: add a 3D pairwise Jastrow (correlation hole at r<R),')
print('   then re-validate E/N and g(r) against the DMC/eGPE reference.')

SS10 sweep summary
------------------------------------------------------------
points (done)        : 7   x in [1.00e-05, 7.99e-03]
inside corridor      : 7/7
stationary           : 5/7
a_eff/a (flat ratio)  : 1.1101 +/- 0.0004

non-stationary points to re-run (more epochs / sampler=pt):
   x=9.280e-05  geweke_z=2.94  epochs=2000
   x=2.827e-04  geweke_z=4.57  epochs=2000

Interpretation:
 - E/N stays inside [4pix, Eq31] -> variationally consistent.
 - E/N / 4pix ~ const -> mean-field regime with renormalised coupling a_eff;
   the LHY sqrt(x) correction is NOT resolved (expected: DeepSet has no pairwise Jastrow).
 - To probe beyond-mean-field: add a 3D pairwise Jastrow (correlation hole at r<R),
   then re-validate E/N and g(r) against the DMC/eGPE reference.
